# Manual Evaluation — Notion Ground-Truth Verification

Loads a dumped evaluation run and verifies each task's real side-effects against the live Notion workspace.

| Task | Verification method |
|---|---|
| `add_task` | Query Tasks DB for title "Add_task_test", assert existence + check importance & date |
| `add_toggle_to_page` | List children of the target page, locate the toggle by title & body text |
| `retrieve_tasks` | Re-query Tasks DB sorted by importance → urgency, compare top-3 with agent output |


## INIT

In [9]:
import os
import sys
import pickle
import glob
from datetime import date, datetime
from pprint import pprint

# ── Make the parent project importable ───────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ── Load .env from project root ───────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

# ── Notion client ─────────────────────────────────────────────────────────────
from notion_client import Client

NOTION_TOKEN               = os.getenv("NOTION_TOKEN")
NOTION_TASKS_DATABASE_ID   = os.getenv("NOTION_TASKS_DATABASE_ID")
NOTION_INBOX_PAGE_ID       = os.getenv("NOTION_INBOX_PAGE_ID")

assert NOTION_TOKEN,             "NOTION_TOKEN not set"
assert NOTION_TASKS_DATABASE_ID, "NOTION_TASKS_DATABASE_ID not set"
assert NOTION_INBOX_PAGE_ID,     "NOTION_INBOX_PAGE_ID not set"

notion = Client(auth=NOTION_TOKEN)
print("Notion client ready.")
print(f"  Tasks DB : {NOTION_TASKS_DATABASE_ID}")
print(f"  Inbox    : {NOTION_INBOX_PAGE_ID}")


Notion client ready.
  Tasks DB : 601d53ff397d42b7bb791577134e51b8
  Inbox    : 24ecb17dcc4480beb3a6e6f0e4751989


In [10]:
# ── Set to a specific .pkl path, or leave as None to auto-load the latest ─────
DUMPED_RESULTS_PATH: str | None = None


def load_results(path: str | None = None) -> dict:
    """
    Load a dumped evaluation run from a .pkl file.

    If ``path`` is None, automatically picks the most-recently created file
    in ``<project_root>/evaluation_results/``.

    Returns a dict with keys:
        "all_pipeline_artifacts"  – {task_id: pipeline artifacts dict}
        "all_eval_results"        – {task_id: evaluation results dict}
    """
    results_dir = os.path.join(PROJECT_ROOT, "evaluation_results")

    if path is None:
        candidates = sorted(glob.glob(os.path.join(results_dir, "*.pkl")))
        if not candidates:
            raise FileNotFoundError(f"No .pkl files found in {results_dir}")
        path = candidates[-1]
        print(f"Auto-selected: {path}")
    else:
        print(f"Loading: {path}")

    with open(path, "rb") as f:
        data = pickle.load(f)

    tasks = list(data.get("all_eval_results", {}).keys())
    print(f"Loaded {len(tasks)} task(s): {tasks}")
    return data


results      = load_results(DUMPED_RESULTS_PATH)
artifacts    = results["all_pipeline_artifacts"]   # task_id → pipeline artifacts
eval_results = results["all_eval_results"]         # task_id → eval scores


Auto-selected: d:\Coding\VSCFiles\IndependentProjects\AI\notion-query\evaluation_results\no_tests_1_iteration_2026-02-27_00-40-04.pkl
Loaded 1 task(s): ['add_task']


In [11]:
# ── Quick summary of loaded scores ───────────────────────────────────────────
print(f"{'Task':<30} {'Overall':>8}  Per-category scores")
print("-" * 80)
for task_id, er in eval_results.items():
    summary = er.get("summary", {})
    overall = summary.get("overall_score", 0)
    cats = "  ".join(
        f"{k.replace('_score', '')}={v:.2f}"
        for k, v in summary.items() if k.endswith("_score") and k != "overall_score"
    )
    passed = artifacts.get(task_id, {}).get("passed", "?")
    print(f"{task_id:<30} {overall:>8.3f}  {cats}  | agent_passed={passed}")


Task                            Overall  Per-category scores
--------------------------------------------------------------------------------
add_task                          0.750  rag=0.75  plan=1.00  tests=0.00  code=1.00  reflection=1.00  | agent_passed=True


---
## Test 1 — `add_task`

**Task**: Add a task titled `"Add_task_test"` with today's date, importance = 4, and project `"Agents & LLMs Specialization"` to the Tasks database.

**Verification**: Query the Tasks database for a page whose `Name` contains `"Add_task_test"` and assert that:
- The page exists
- `Importance` = 4
- `Due Date` = today


In [ ]:
TASK_ID = "add_task"
EXPECTED_TITLE      = "Add_task_test"
EXPECTED_IMPORTANCE = 4
EXPECTED_DATE       = date.today().isoformat()   # "YYYY-MM-DD"

# ── Pull the agent's final generated code for reference ──────────────────────
agent_code = artifacts.get(TASK_ID, {}).get("final_code", "(no code in artifacts)")
agent_passed = artifacts.get(TASK_ID, {}).get("passed", False)
print(f"Agent passed tests: {agent_passed}")
print("─" * 60)
print(agent_code[:600] if agent_code else "(empty)")
print("─" * 60)

# ── Query Tasks data source for the title ────────────────────────────────────
response = notion.data_sources.query(
    data_source_id=NOTION_TASKS_DATABASE_ID,
    filter={
        "property": "Name",
        "title": {"contains": EXPECTED_TITLE},
    },
)
pages = response.get("results", [])
print(f"\nNotion found {len(pages)} page(s) matching '{EXPECTED_TITLE}'")

# ── Assertions ────────────────────────────────────────────────────────────────
assert len(pages) > 0, f"FAIL — no page with title '{EXPECTED_TITLE}' found in Tasks data source"

# Use the most recently created match
page = pages[0]
page_id    = page["id"]
props      = page["properties"]

# Title
title_text = props["Name"]["title"][0]["plain_text"] if props["Name"]["title"] else ""
print(f"\n  title      : {title_text!r}")
assert EXPECTED_TITLE in title_text, f"FAIL — title mismatch: {title_text!r}"

# Importance
importance = props.get("Importance", {}).get("number")
print(f"  importance : {importance}")
assert importance == EXPECTED_IMPORTANCE, f"FAIL — importance={importance}, expected {EXPECTED_IMPORTANCE}"

# Due Date
due_date = (props.get("Due", {}).get("date") or
            props.get("Due Date", {}).get("date") or
            props.get("Date", {}).get("date") or {})
due_start = due_date.get("start", "") if due_date else ""
print(f"  due date   : {due_start!r}  (expected: {EXPECTED_DATE!r})")
assert due_start == EXPECTED_DATE, f"FAIL — date={due_start!r}, expected {EXPECTED_DATE!r}"

print(f"\nPASS — page '{EXPECTED_TITLE}' verified in Notion  (id={page_id})")


---
## Test 2 — `add_toggle_to_page`

**Task**: Add a toggle block titled `"Add_toggle_test"` with child content `"This is a test toggle"` to page `24ecb17dcc4480beb3a6e6f0e4751989`.

**Verification**: List the children of that page, find a toggle whose rich text matches the expected title, then inspect its children for the expected body text.


In [ ]:
TASK_ID          = "add_toggle_to_page"
TARGET_PAGE_ID   = "24ecb17dcc4480beb3a6e6f0e4751989"
EXPECTED_TITLE   = "Add_toggle_test"
EXPECTED_CONTENT = "This is a test toggle"

# ── Pull the agent's final generated code for reference ──────────────────────
agent_code   = artifacts.get(TASK_ID, {}).get("final_code", "(no code in artifacts)")
agent_passed = artifacts.get(TASK_ID, {}).get("passed", False)
print(f"Agent passed tests: {agent_passed}")
print("─" * 60)
print(agent_code[:600] if agent_code else "(empty)")
print("─" * 60)

# ── Helpers ───────────────────────────────────────────────────────────────────
def rich_text_to_str(rich_text: list) -> str:
    return "".join(rt.get("plain_text", "") for rt in rich_text)

def get_all_children(block_id: str) -> list:
    """Fetch all child blocks (handles pagination)."""
    children, cursor = [], None
    while True:
        kwargs = {"block_id": block_id}
        if cursor:
            kwargs["start_cursor"] = cursor
        resp = notion.blocks.children.list(**kwargs)
        children.extend(resp.get("results", []))
        if not resp.get("has_more"):
            break
        cursor = resp.get("next_cursor")
    return children

# ── List children of the target page ─────────────────────────────────────────
page_children = get_all_children(TARGET_PAGE_ID)
print(f"\nPage has {len(page_children)} top-level block(s)")

# ── Find matching toggle ──────────────────────────────────────────────────────
toggle_block = None
for block in page_children:
    if block.get("type") != "toggle":
        continue
    toggle_text = rich_text_to_str(block["toggle"].get("rich_text", []))
    if EXPECTED_TITLE in toggle_text:
        toggle_block = block
        print(f"  Found toggle: {toggle_text!r}  (id={block['id']})")
        break

assert toggle_block is not None, (
    f"FAIL — no toggle block containing '{EXPECTED_TITLE}' found on page {TARGET_PAGE_ID}"
)

# ── Check toggle children for expected content ────────────────────────────────
toggle_children = get_all_children(toggle_block["id"])
print(f"  Toggle has {len(toggle_children)} child block(s)")

found_content = False
for child in toggle_children:
    child_type = child.get("type", "")
    child_rt   = child.get(child_type, {}).get("rich_text", [])
    child_text = rich_text_to_str(child_rt)
    print(f"    [{child_type}] {child_text!r}")
    if EXPECTED_CONTENT in child_text:
        found_content = True

assert found_content, (
    f"FAIL — toggle content '{EXPECTED_CONTENT}' not found in toggle children"
)
print(f"\nPASS — toggle '{EXPECTED_TITLE}' with correct content verified in Notion")


---
## Test 3 — `retrieve_tasks`

**Task**: Retrieve the last 3 tasks from the Tasks database sorted by Importance (desc) then Urgency (desc).

**Verification**: Query the Tasks database with the same sort, take the top 3, and compare titles/IDs with the agent's captured `solution_run` stdout output.


In [12]:
from notion_client.errors import APIResponseError

TASK_ID    = "retrieve_tasks"
TASK_LIMIT = 3

# ── Pull agent artifacts ──────────────────────────────────────────────────────
task_artifacts = artifacts.get(TASK_ID, {})
agent_passed   = task_artifacts.get("passed", False)
agent_code     = task_artifacts.get("final_code", "(no code in artifacts)")
trials         = task_artifacts.get("trials", [])

print(f"Agent passed tests: {agent_passed}")
print("─" * 60)
print(agent_code[:600] if agent_code else "(empty)")
print("─" * 60)

# Print what the agent's solution.py actually printed to stdout (last trial)
if trials:
    last_sol_run = trials[-1].get("solution_run", {})
    print(f"\nAgent solution.py stdout:\n{last_sol_run.get('stdout', '(empty)')}")

# ── Ground truth: query Tasks data source with the same sort ─────────────────
try:
    # Use the Data Sources API (NOTION_TASKS_DATABASE_ID is a data source ID)
    response = notion.data_sources.query(
        data_source_id=NOTION_TASKS_DATABASE_ID,
        sorts=[
            {"property": "Importance", "direction": "descending"},
            {"property": "Urgency",    "direction": "descending"},
        ],
        page_size=TASK_LIMIT,
    )
    gt_pages = response.get("results", [])
except APIResponseError as e:
    gt_pages = []
    print("\nWARN — Could not query the Tasks data source.")
    print(f"Reason: {e}")
    print("Make sure this data source is shared with your integration:")
    print(f"  data_source_id={NOTION_TASKS_DATABASE_ID}")

print(f"\nGround truth — top {TASK_LIMIT} tasks (Importance ↓, Urgency ↓):")
gt_ids    = []
gt_titles = []
for i, page in enumerate(gt_pages, 1):
    title_rt = page["properties"].get("Name", {}).get("title", [])
    title    = title_rt[0]["plain_text"] if title_rt else "(untitled)"
    imp      = page["properties"].get("Importance", {}).get("number")
    urg      = page["properties"].get("Urgency",    {}).get("number")
    print(f"  {i}. {title!r}  importance={imp}  urgency={urg}  id={page['id']}")
    gt_ids.append(page["id"])
    gt_titles.append(title)

# ── Compare with agent output ─────────────────────────────────────────────────
# The agent's result is intentionally flexible — we check whether the returned
# page IDs appear anywhere in the agent's stdout or final code.
agent_stdout = (trials[-1].get("solution_run", {}).get("stdout", "") if trials else "")
agent_output_combined = agent_stdout + agent_code

print("\nID match summary:")
all_matched = True
for pid, ptitle in zip(gt_ids, gt_titles):
    # Strip hyphens for a more lenient match (Notion returns both formats)
    pid_clean = pid.replace("-", "")
    matched   = pid_clean in agent_output_combined.replace("-", "")
    status    = "MATCH " if matched else "MISS  "
    if not matched:
        all_matched = False
    print(f"  {status} {ptitle!r}  ({pid})")

if all_matched:
    print(f"\nPASS — all {TASK_LIMIT} expected tasks found in agent output")
else:
    print("\nWARN — some task IDs were not found in agent output; inspect manually above")


Agent passed tests: False
────────────────────────────────────────────────────────────
(no code in artifacts)
────────────────────────────────────────────────────────────

WARN — Could not query the Tasks data source.
Reason: Could not find database with ID: 601d53ff-397d-42b7-bb79-1577134e51b8. Make sure the relevant pages and databases are shared with your integration.
Make sure this data source is shared with your integration:
  data_source_id=601d53ff397d42b7bb791577134e51b8

Ground truth — top 3 tasks (Importance ↓, Urgency ↓):

ID match summary:

PASS — all 3 expected tasks found in agent output


---
## Overall Ground-Truth Score


In [ ]:
# ── Aggregate LLM eval scores for every loaded task ──────────────────────────
print(f"{'Task':<30} {'Agent':>6}  {'RAG':>6}  {'Plan':>6}  {'Tests':>6}  {'Code':>6}  {'Refl':>6}  {'Overall':>8}")
print("─" * 90)
for task_id, er in eval_results.items():
    s        = er.get("summary", {})
    passed   = "✓" if artifacts.get(task_id, {}).get("passed") else "✗"
    rag      = s.get("rag_score", 0)
    plan     = s.get("plan_score", 0)
    tests    = s.get("tests_score", 0)
    code     = s.get("code_score", 0)
    refl     = s.get("reflection_score", 0)
    overall  = s.get("overall_score", 0)
    print(f"{task_id:<30} {passed:>6}  {rag:>6.2f}  {plan:>6.2f}  {tests:>6.2f}  {code:>6.2f}  {refl:>6.2f}  {overall:>8.3f}")


---
## Raw Data Retrieval — Projects & Tasks (JSON)

Quick snapshot of live Notion data. Fetches a sample of pages from both the
Projects and Tasks data sources and prints them as formatted JSON for inspection.


In [13]:
import json

NOTION_PROJECTS_DATA_SOURCE_ID = os.getenv("NOTION_PROJECTS_DATA_SOURCE_ID")
NOTION_TASKS_DATA_SOURCE_ID    = os.getenv("NOTION_TASKS_DATA_SOURCE_ID")

SAMPLE_SIZE = 5  # number of pages to retrieve from each data source


def _extract_plain_text(rich_text_list: list) -> str:
    return "".join(rt.get("plain_text", "") for rt in rich_text_list)


def page_to_simple_dict(page: dict) -> dict:
    """Flatten a Notion page object into a plain, JSON-serialisable dict."""
    props = page.get("properties", {})
    out = {"id": page["id"], "url": page.get("url", "")}
    for prop_name, prop_val in props.items():
        ptype = prop_val.get("type", "")
        if ptype == "title":
            out[prop_name] = _extract_plain_text(prop_val.get("title", []))
        elif ptype == "rich_text":
            out[prop_name] = _extract_plain_text(prop_val.get("rich_text", []))
        elif ptype in ("number", "checkbox"):
            out[prop_name] = prop_val.get(ptype)
        elif ptype == "select":
            sel = prop_val.get("select")
            out[prop_name] = sel["name"] if sel else None
        elif ptype == "multi_select":
            out[prop_name] = [s["name"] for s in prop_val.get("multi_select", [])]
        elif ptype == "date":
            d = prop_val.get("date")
            out[prop_name] = d["start"] if d else None
        elif ptype == "status":
            st = prop_val.get("status")
            out[prop_name] = st["name"] if st else None
        else:
            out[prop_name] = f"<{ptype}>"
    return out


# ── Retrieve Projects ─────────────────────────────────────────────────────────
print("=" * 60)
print(f"PROJECTS  (top {SAMPLE_SIZE})")
print("=" * 60)
if NOTION_PROJECTS_DATA_SOURCE_ID:
    proj_resp = notion.data_sources.query(
        data_source_id=NOTION_PROJECTS_DATA_SOURCE_ID,
        page_size=SAMPLE_SIZE,
    )
    projects = [page_to_simple_dict(p) for p in proj_resp.get("results", [])]
    print(json.dumps(projects, indent=2, ensure_ascii=False))
else:
    print("NOTION_PROJECTS_DATA_SOURCE_ID not set — skipping.")

# ── Retrieve Tasks ────────────────────────────────────────────────────────────
print()
print("=" * 60)
print(f"TASKS  (top {SAMPLE_SIZE}, sorted by Importance ↓ then Urgency ↓)")
print("=" * 60)
tasks_source_id = NOTION_TASKS_DATA_SOURCE_ID or NOTION_TASKS_DATABASE_ID
task_resp = notion.data_sources.query(
    data_source_id=tasks_source_id,
    sorts=[
        {"property": "Importance", "direction": "descending"},
        {"property": "Urgency",    "direction": "descending"},
    ],
    page_size=SAMPLE_SIZE,
)
tasks = [page_to_simple_dict(p) for p in task_resp.get("results", [])]
print(json.dumps(tasks, indent=2, ensure_ascii=False))


PROJECTS  (top 5)
[
  {
    "id": "06457eab-945b-407c-aa5b-93bfa6ef7c55",
    "url": "https://www.notion.so/Leads-06457eab945b407caa5b93bfa6ef7c55",
    "Blocked by": "<relation>",
    "Timing": null,
    "Created": "<created_time>",
    "Blocking": "<relation>",
    "Status": "Not started",
    "Type": [
      "Self-Improvement"
    ],
    "Task left": "<formula>",
    "Related projects": "<relation>",
    "Preview": "<files>",
    "Tasks": "<relation>",
    "Priority": "🟡",
    "Remaining days": "<formula>",
    "Task Counter": "<rollup>",
    "Name": "Leads",
    "Urgency": null,
    "Importance": null,
    "Due Date": null
  },
  {
    "id": "072d966c-2c11-4d84-a052-aa9cff11a202",
    "url": "https://www.notion.so/Gitlab-072d966c2c114d84a052aa9cff11a202",
    "Blocked by": "<relation>",
    "Timing": null,
    "Created": "<created_time>",
    "Blocking": "<relation>",
    "Status": "Archive",
    "Type": [
      "Learning"
    ],
    "Task left": "<formula>",
    "Related projects"